# 05 - model training (preliminary experiments)

train mfcc + svm baseline on sample data. get real numbers for midterm report.

In [ ]:
import time, joblib
import pandas as pd
import numpy as np
import librosa
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from src.config.settings import SAMPLE_RATE, N_MFCC, N_FFT, HOP_LENGTH

SAMPLES = Path("../../data/samples")
df = pd.read_csv(SAMPLES / "sample_labels.csv")
emotion_map = {1:'neutral',2:'calm',3:'happy',4:'sad',5:'angry',6:'fearful',7:'disgust',8:'surprised'}
print(f"{len(df)} files loaded")

## 1. extract mfcc features

In [ ]:
def extract_mfcc(fp):
    y, sr = librosa.load(fp, sr=SAMPLE_RATE)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    d = librosa.feature.delta(m)
    d2 = librosa.feature.delta(m, order=2)
    return np.concatenate([m.mean(1), m.std(1), d.mean(1), d.std(1), d2.mean(1), d2.std(1)])

train = df[df['split']=='train']
test = df[df['split']=='test']
print(f"train: {len(train)}, test: {len(test)}")

t0 = time.time()
X_train = np.array([extract_mfcc(fp) for fp in train['filepath']])
X_test = np.array([extract_mfcc(fp) for fp in test['filepath']])
y_train, y_test = train['emotion_code'].values, test['emotion_code'].values
t1 = time.time()
print(f"extraction: {t1-t0:.1f}s for {len(train)+len(test)} clips")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. train svm

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

t0 = time.time()
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train_s, y_train)
t1 = time.time()
print(f"training: {t1-t0:.2f}s")

y_pred = svm.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f"\ntest accuracy: {acc*100:.1f}%\n")
print(classification_report(y_test, y_pred, target_names=list(emotion_map.values()), digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(emotion_map.values()),
            yticklabels=list(emotion_map.values()))
plt.title('svm confusion matrix (sample test)')
plt.tight_layout()
plt.savefig(str(SAMPLES / 'svm_confusion_sample.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f"correct: {np.trace(cm)}/{len(y_test)} ({np.trace(cm)/len(y_test)*100:.1f}%)")

## 3. timing + model export

In [ ]:
import platform
n_runs = 50
single = X_test_s[0:1]
t0 = time.time()
for _ in range(n_runs):
    svm.predict(single)
t1 = time.time()
single_ms = (t1-t0)/n_runs*1000

batch = X_test_s[:32]
t0 = time.time()
for _ in range(n_runs):
    svm.predict(batch)
t1 = time.time()
batch_ms = (t1-t0)/n_runs*1000
print(f"inference: {single_ms:.2f}ms (single), {batch_ms:.2f}ms (batch 32)")

In [ ]:
model_path = SAMPLES / 'svm_baseline_sample.pkl'
joblib.dump(svm, model_path)
size_kb = model_path.stat().st_size / 1024
print(f"model: {size_kb:.1f} kb")

## results (for midterm report)

fill into 05-preliminary-experiments-and-results.md:
- test accuracy: x.x%
- total clips: x
- extraction time: x.xs for x clips
- training time: x.xs
- single inference: x.xxms
- model size: x.x kb